### __Django REST API – CRUD with DRF__

Django REST Framework is used to create web APIs very easily and efficiently. This is a wrapper around the Django Framework. There are three stages before creating an API through the REST framework, Converting a Model’s data to JSON/XML format (Serialization), Rendering this data to the view, and Creating a URL for mapping to the views.

* Install Django REST Framework

```pip install djangorestframework```

In [ ]:
INSTALLED_APPS = [
	...,
	'rest_framework', 
]

---

### Serializing Django Objects

Serializers in Django REST Framework converts the objects into data types that are understandable by javascript and front-end frameworks. Serializers also provide deserialization, allowing parsed data to be converted back into complex types, after first validating the incoming data. The two major serializers that are most popularly used are ModelSerializer and HyperLinkedModelSerialzer.

    [Serializers – Django REST Framework](https://www.geeksforgeeks.org/serializers-django-rest-framework/?ref=gcse)  
[HyperlinkedModelSerializer in serializers – Django REST Framework](https://www.geeksforgeeks.org/hyperlinkedmodelserializer-in-serializers-django-rest-framework/)

ModelSerializer is a layer of abstraction over the default serializer that allows to quickly create a serializer for a model in Django. It provides a shortcut that lets you automatically create a Serializer class with fields that correspond to the Model fields.

* Now let’s create our ```serlializers.py``` file in the api folder and add the below code

In [ ]:
from rest_framework import serializers
from .models import Item

class ItemSerializer(serializers.ModelSerializer):
	class Meta:
		model = Item
		fields = ('category', 'subcategory', 'name', 'amount')


* Create Views for Django

In [ ]:
from rest_framework.decorators import api_view
from rest_framework.response import Response
from .models import Item
from .serializers import ItemSerializer

@api_view(['GET'])
def ApiOverview(request):
	api_urls = {
		'all_items': '/',
		'Search by Category': '/?category=category_name',
		'Search by Subcategory': '/?subcategory=category_name',
		'Add': '/create',
		'Update': '/update/pk',
		'Delete': '/item/pk/delete'
	}

	return Response(api_urls)


In the above code, the api_view decorator takes a list of HTTP methods that a views should response to. Other methods will response with the Method __Not Allowed__.

* urls.py

In [ ]:
from django.urls import path
from . import views

urlpatterns = [
	path('', views.ApiOverview, name='home'),
]


* 127.0.0.1:8000

HTTP 200 OK  
Allow: GET, OPTIONS  
Content-Type: application/json  
Vary: Accept  
  
{  
    "all_items": "/",  
    "Search by Category": "/?category=category_name",  
    "Search by Subcategory": "/?subcategory=category_name",  
    "Add": "/create",  
    "Update": "/update/pk",  
    "Delete": "/item/pk/delete"  
}  

---

### Django Rest Framework – Create View

Now our create view will use the POST method for inserting data into our database. Let’s create our add_items function in the views.py file.

* views.py

In [ ]:
from rest_framework import serializers
from rest_framework import status

@api_view(['POST'])
def add_items(request):
	item = ItemSerializer(data=request.data)

	# validating for already existing data
	if Item.objects.filter(**request.data).exists():
		raise serializers.ValidationError('This data already exists')

	if item.is_valid():
		item.save()
		return Response(item.data)
	else:
		return Response(status=status.HTTP_404_NOT_FOUND)


* urls.py

In [ ]:
from django.urls import path
from . import views

urlpatterns = [
	path('', views.ApiOverview, name='home'),
	path('create/', views.add_items, name='add-items'),
]

---

### Django Rest Framework – Read View

* views.py

In [ ]:
@api_view(['GET'])
def view_items(request):
	
	
	# checking for the parameters from the URL
	if request.query_params:
		items = Item.objects.filter(**request.query_params.dict())
	else:
		items = Item.objects.all()

	# if there is something in items else raise error
	if items:
		serializer = ItemSerializer(items, many=True)
		return Response(serializer.data)
	else:
		return Response(status=status.HTTP_404_NOT_FOUND)


This view function also lets us filter by category or subcategory. You can use either of the following URLs i.e. http://127.0.0.1:8000/api/?category=category_name or http://127.0.0.1:8000/api/?subcategory=category_name to filter for both category and subcategory respectively. You can also use http://127.0.0.1:8000/api/all/?name=item_name to search for a specific item.

* urls.py

In [ ]:
from django.urls import path
from . import views

urlpatterns = [
	path('', views.ApiOverview, name='home'),
	path('create/', views.add_items, name='add-items'),
	path('all/', views.view_items, name='view_items'),

]

---

### Django Rest Framework – Update View

* views.py

In [ ]:
@api_view(['POST'])
def update_items(request, pk):
	item = Item.objects.get(pk=pk)
	data = ItemSerializer(instance=item, data=request.data)

	if data.is_valid():
		data.save()
		return Response(data.data)
	else:
		return Response(status=status.HTTP_404_NOT_FOUND)


* urls.py

In [ ]:
from django.urls import path
from . import views

urlpatterns = [
	path('', views.ApiOverview, name='home'),
	path('create/', views.add_items, name='add-items'),
	path('all/', views.view_items, name='view_items'),
	path('update/<int:pk>/', views.update_items, name='update-items'),

]

-----

### Django Rest Framework – Delete View

views.py

In [ ]:
from django.utils import get_object_or_404

@api_view(['DELETE'])
def delete_items(request, pk):
	item = get_object_or_404(Item, pk=pk)
	item.delete()
	return Response(status=status.HTTP_202_ACCEPTED)


* urls.py

In [ ]:
from django.urls import path
from . import views

urlpatterns = [
	path('', views.ApiOverview, name='home'),
	path('create/', views.add_items, name='add-items'),
	path('all/', views.view_items, name='view_items'),
	path('update/<int:pk>/', views.update_items, name='update-items'),
	path('item/<int:pk>/delete/', views.delete_items, name='delete-items'),
	
]

---

### Request

__data__  
`request.data` returns the parsed content of the request body. This is similar to the standard `request.POST` and `request.FILES` attributes except that:
* It includes all parsed content, including file and non-file inputs.  
* It supports parsing the content of HTTP methods other than POST, meaning that you can access the content of PUT and PATCH requests.

__query_params__  
  
`request.query_params` is a more correctly named synonym for `request.GET`.  
For clarity inside your code, we recommend using request.query_params instead of the Django's standard request.GET. Doing so will help keep your codebase more correct and obvious - any HTTP method type may include query parameters, not just GET requests.

__request.parsers__
  
The `APIView` class or `@api_view` decorator will ensure that this property is automatically set to a list of `Parser` instances, based on the `parser_classes` set on the view or based on the `DEFAULT_PARSER_CLASSES` setting.

You won't typically need to access this property.

__Authentication__  
  
`request.user` typically returns an instance of `django.contrib.auth.models.User`, although the behavior depends on the authentication policy being used.  
    
`request.auth` returns any additional authentication context. The exact behavior of `request.auth` depends on the authentication policy being used, but it may typically be an instance of the token that the request was authenticated against.

If the request is unauthenticated, or if no additional context is present, the default value of `request.auth` is `None`.

---

### Exmaple - User Registration

* models.py

In [ ]:
from django.db import models

class Employee(models.Model):
    forename = models.CharField(max_length=250)
    surname = models.CharField(max_length=250)
    email = models.EmailField()
    age = models.PositiveSmallIntegerField()

    def __str__(self):
        return f"{self.forename} - {self.email} - {self.age}"

* serializers.py

In [ ]:
from rest_framework import serializers

from .models import Employee

class EmployeeRegisterSerializer(serializers.ModelSerializer):
    class Meta:
        model = Employee
        fields = ["forename", "surname", "email", "age"]

    def create(self, validated_data):
        del validated_data["confirm_password"]
        return Employee.objects.create(**validated_data)

* views.py


In [ ]:
from rest_framework.views import APIView
from rest_framework.response import Response

from .models import Employee
from .serializers import EmployeeRegisterSerializer

class UserRegisterView(APIView):
    def post(self, request):
        ser_data = EmployeeRegisterSerializer(data=request.POST)
        if ser_data.is_valid():
            ser_data.create(ser_data.validated_data)
            return Response(ser_data.data)
        return Response(ser_data.errors)


---

### Custom Validation

In [ ]:
from rest_framework import serializers
from django.contrib.auth.models import User

class UserRegisterSerializer(serializers.ModelSerializer):
    confirm_password = serializers.CharField(require=True, write_only=True)

    class Meta:
        model = User
        fields = ["username", "email", "password", "confirm_password"]
        extra_kwargs = {
            "password": {"write_only": True}
        }

    def validate(self, data):
        if data["password"] != data["confirm_password"]:
            raise serializers.ValidationError("Password must be match")
        return data

---

### Authentication

* The` request.user` property will typically be set to an instance of the `contrib.auth` package's `User` class.

* The `request.auth` property is used for any additional authentication information, for example, it may be used to represent an authentication token that the request was signed with.

__Note__: Don't forget that authentication by itself won't allow or disallow an incoming request, it simply identifies the credentials that the request was made with.

_How authentication is determined_

+ The default authentication schemes may be set globally, using the DEFAULT_AUTHENTICATION_CLASSES setting.

In [ ]:
REST_FRAMEWORK = {
    'DEFAULT_AUTHENTICATION_CLASSES': [
        'rest_framework.authentication.BasicAuthentication',
        'rest_framework.authentication.SessionAuthentication',
    ]
}

* You can also set the authentication scheme on a per-view or per-viewset basis, using the APIView class-based views.

In [ ]:
from rest_framework.authentication import SessionAuthentication, BasicAuthentication
from rest_framework.permissions import IsAuthenticated
from rest_framework.views import APIView

class ExampleView(APIView):
    authentication_classes = [SessionAuthentication, BasicAuthentication]
    permission_classes = [IsAuthenticated]

* Or, if you're using the @api_view decorator with function based views.

In [ ]:
@api_view(['GET'])
@authentication_classes([SessionAuthentication, BasicAuthentication])
@permission_classes([IsAuthenticated])
def example_view(request, format=None):
    ...

##### TokenAuthentication  
__Note__: The token authentication provided by Django REST framework is a fairly simple implementation.

To use the `TokenAuthentication` scheme you'll need to configure the authentication classes to include `TokenAuthentication`, and additionally `include rest_framework.authtoken` in your `INSTALLED_APPS` setting:

In [ ]:
# Rest Framework
INSTALLED_APPS = [
    ...,
    'rest_framework.authtoken'
]

Make sure to run `manage.py migrate` after changing your settings.  
The `rest_framework.authtoken` app provides Django database migrations.

When using `TokenAuthentication`, you may want to provide a mechanism for clients to obtain a token given the username and password. REST framework provides a built-in view to provide this behavior. To use it, add the `obtain_auth_token` view to your URLconf:

In [ ]:
from rest_framework.authtoken import views

urlpatterns += [
    path('api-token-auth/', views.obtain_auth_token)
]

Note that the URL part of the pattern can be whatever you want to use.  
  
The `obtain_auth_token` view will return a JSON response when valid username and password fields are POSTed to the view using form data or JSON:  
  
`{ 'token' : '9944b09199c62bcf9418ad846dd0e4bbdfc6ee4b' }`

***

### Permissions  
  
Permission checks are always run at the very start of the view, before any other code is allowed to proceed. Permission checks will typically use the authentication information in the `request.user` and `request.auth` properties to determine if the incoming request should be permitted.

_How permissions are determined_

* Setting the permission policy  
The default permission policy may be set globally, using the `DEFAULT_PERMISSION_CLASSES` setting.

In [ ]:
REST_FRAMEWORK = {
    'DEFAULT_PERMISSION_CLASSES': [
        'rest_framework.permissions.IsAuthenticated',
    ]
}

* You can also set the authentication policy on a per-view, or per-viewset basis, using the APIView class-based views.

In [ ]:
from rest_framework.permissions import IsAuthenticated
from rest_framework.views import APIView

class ExampleView(APIView):
    permission_classes = [IsAuthenticated]

##### _API Reference_

__AllowAny__  
  
The `AllowAny` permission class will allow unrestricted access, regardless of if the request was authenticated or unauthenticated.  

This permission is not strictly required, since you can achieve the same result by using an empty list or tuple for the permissions setting, but you may find it useful to specify this class because it makes the intention explicit.

__IsAuthenticated__

The `IsAuthenticated` permission class will deny permission to any unauthenticated user, and allow permission otherwise.

This permission is suitable if you want your API to only be accessible to registered users.

__IsAdminUser__

The `IsAdminUser` permission class will deny permission to any user, unless `user.is_staff` is `True` in which case permission will be allowed.

This permission is suitable if you want your API to only be accessible to a subset of trusted administrators.

__IsAuthenticatedOrReadOnly__

The `IsAuthenticatedOrReadOnly` will allow authenticated users to perform any request. Requests for unauthenticated users will only be permitted if the request method is one of the "safe" methods; `GET`, `HEAD` or `OPTIONS`.

This permission is suitable if you want to your API to allow read permissions to anonymous users, and only allow write permissions to authenticated users.

***

### Throttling
  
Throttling is similar to permissions, in that it determines if a request should be authorized. Throttles indicate a temporary state, and are used to control the rate of requests that clients can make to an API.  
  
_The application-level throttling that REST framework provides should not be considered a security measure or protection againstbrute forcing or denial-of-service attacks. Deliberately malicious actors will always be able to spoof IP origins. In addition to this, the built-in throttling implementations are implemented using Django's cache framework, and use non-atomic operations to determine the request rate, which may sometimes result in some fuzziness._

##### __How throttling is determined__

* Setting the throttling policy  
The default throttling policy may be set globally, using the `DEFAULT_THROTTLE_CLASSES` and `DEFAULT_THROTTLE_RATES` settings.

In [ ]:
REST_FRAMEWORK = {
    'DEFAULT_THROTTLE_CLASSES': [
        'rest_framework.throttling.AnonRateThrottle',
        'rest_framework.throttling.UserRateThrottle'
    ],
    'DEFAULT_THROTTLE_RATES': {
        'anon': '100/day',
        'user': '1000/day'
    }
}

The rate descriptions used in DEFAULT_THROTTLE_RATES may include `second`, `minute`, `hour` or `day` as the throttle period.

* You can also set the throttling policy on a per-view or per-viewset basis, using the APIView class-based views.

In [ ]:
from rest_framework.throttling import UserRateThrottle
from rest_framework.views import APIView

class ExampleView(APIView):
    throttle_classes = [UserRateThrottle]

***

### Parsers  
  
REST framework includes a number of built-in Parser classes, that allow you to accept requests with various media types. There is also support for defining your own custom parsers, which gives you the flexibility to design the media types that your API accepts.  
  
__Note__: When developing client applications always remember to make sure you're setting the Content-Type header when sending data in an HTTP request.

##### _Setting the parsers_ 
   
Parses JSON request content. `request.data` will be populated with a dictionary of data.
.media_type: `application/json`


* The default set of parsers may be set globally, using the `DEFAULT_PARSER_CLASSES` setting. For example, the following settings would allow only requests with `JSON` content, instead of the default of JSON or form data.

In [ ]:
REST_FRAMEWORK = {
    'DEFAULT_PARSER_CLASSES': [
        'rest_framework.parsers.JSONParser',
    ]
}

* You can also set the parsers used for an individual view, or viewset, using the `APIView` class-based views.

In [ ]:
from rest_framework.parsers import JSONParser
from rest_framework.views import APIView

class ExampleView(APIView):
    """
    A view that can accept POST requests with JSON content.
    """
    parser_classes = [JSONParser]

***

### Renderers  
  
REST framework includes a number of built in Renderer classes, that allow you to return responses with various media types. There is also support for defining your own custom renderers, which gives you the flexibility to design your own media types.

*  _How the renderer is determined_  
The set of valid renderers for a view is always defined as a list of classes. When a view is entered REST framework will perform content negotiation on the incoming request, and determine the most appropriate renderer to satisfy the request.

* Setting the renderers  
The default set of renderers may be set globally, using the DEFAULT_RENDERER_CLASSES setting. For example, the following settings would use JSON as the main media type and also include the self describing API.

In [ ]:
REST_FRAMEWORK = {
    'DEFAULT_RENDERER_CLASSES': [
        'rest_framework.renderers.JSONRenderer',
        'rest_framework.renderers.BrowsableAPIRenderer',
    ]
}

* You can also set the renderers used for an individual view, or viewset, using the `APIView` class-based views.

In [ ]:
from django.contrib.auth.models import User
from rest_framework.renderers import JSONRenderer
from rest_framework.views import APIView

class UserCountView(APIView):
    """
    A view that returns the count of active users in JSON.
    """
    renderer_classes = [JSONRenderer]

***